In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("RDD Exercises").getOrCreate()
sc = spark.sparkContext
print("Spark Ready")

Spark Ready


In [7]:
from pyspark import SparkContext

logs = [
  "Rahul login", "Sneha login", "Arjun login",
  "Rahul purchase", "Priya login", "Sneha logout",
  "Rahul login", "Karan login", "Arjun purchase",
  "Sneha login", "Rahul logout", "Amit login",
  "Priya purchase", "Karan purchase", "Sneha login",
  "Rahul purchase", "Meera login", "Arjun login",
  "Sneha purchase", "Rahul login"
]
logs_rdd = sc.parallelize(logs)

sales = [
  ("Laptop",1,75000), ("Mouse",3,500),
  ("Keyboard",2,1500), ("Laptop",1,75000),
  ("Monitor",1,12000), ("Mouse",2,500),
  ("Keyboard",1,1500), ("Laptop",1,75000),
  ("Monitor",2,12000), ("Mouse",3,500),
  ("Laptop",1,75000), ("Keyboard",2,1500),
  ("Monitor",1,12000), ("Mouse",1,500),
  ("Laptop",1,75000)
]
sales_rdd = sc.parallelize(sales)

employees = [
  ("Rahul","IT",70000), ("Sneha","HR",60000),
  ("Arjun","IT",75000), ("Priya","Finance",80000),
  ("Karan","IT",50000), ("Amit","HR",58000),
  ("Meera","Finance",82000), ("Ravi","IT",72000),
  ("Neha","HR",61000), ("Vikram","Finance",90000)
]
emp_rdd = sc.parallelize(employees)

In [8]:
# Part 1
total = logs_rdd.count()
print(total)

20


In [10]:
first5 = logs_rdd.take(5)
print(first5)

['Rahul login', 'Sneha login', 'Arjun login', 'Rahul purchase', 'Priya login']


In [48]:
users = logs_rdd.map(lambda x: x.split()[0])
print(users.collect())

['Rahul', 'Sneha', 'Arjun', 'Rahul', 'Priya', 'Sneha', 'Rahul', 'Karan', 'Arjun', 'Sneha', 'Rahul', 'Amit', 'Priya', 'Karan', 'Sneha', 'Rahul', 'Meera', 'Arjun', 'Sneha', 'Rahul']


In [13]:
actions = logs_rdd.map(lambda x: x.split()[1])
print(actions.collect())

['login', 'login', 'login', 'purchase', 'login', 'logout', 'login', 'login', 'purchase', 'login', 'logout', 'login', 'purchase', 'purchase', 'login', 'purchase', 'login', 'login', 'purchase', 'login']


In [12]:
unique_users = logs_rdd.map(lambda x: x.split()[0]).distinct()
print(unique_users.collect())

['Karan', 'Amit', 'Rahul', 'Sneha', 'Arjun', 'Priya', 'Meera']


In [14]:
login_count = logs_rdd.filter(lambda x: x.split()[1] == "login").count()
print(login_count)

12


In [15]:
purchases = logs_rdd.filter(lambda x: x.split()[1] == "purchase")
print(purchases.collect())

['Rahul purchase', 'Arjun purchase', 'Priya purchase', 'Karan purchase', 'Rahul purchase', 'Sneha purchase']


In [16]:
# Part 2
sentences = sc.parallelize([
  "big data spark", "spark hadoop hive",
  "python spark hadoop", "spark streaming kafka"
])

In [17]:
words = sentences.flatMap(lambda x: x.split())
print(words.collect())

['big', 'data', 'spark', 'spark', 'hadoop', 'hive', 'python', 'spark', 'hadoop', 'spark', 'streaming', 'kafka']


In [18]:
distinct_tech = sentences.flatMap(lambda x: x.split()).distinct()
print(distinct_tech.collect())

['big', 'hadoop', 'hive', 'python', 'streaming', 'kafka', 'data', 'spark']


In [19]:
word_count = (sentences
  .flatMap(lambda x: x.split())
  .map(lambda w: (w, 1))
  .reduceByKey(lambda a, b: a + b)
)
print(word_count.collect())

[('big', 1), ('hadoop', 2), ('hive', 1), ('python', 1), ('streaming', 1), ('kafka', 1), ('data', 1), ('spark', 4)]


In [20]:
# Part 3
revenue_per_row = sales_rdd.map(lambda x: (x[0], x[1] * x[2]))
print(revenue_per_row.collect())

[('Laptop', 75000), ('Mouse', 1500), ('Keyboard', 3000), ('Laptop', 75000), ('Monitor', 12000), ('Mouse', 1000), ('Keyboard', 1500), ('Laptop', 75000), ('Monitor', 24000), ('Mouse', 1500), ('Laptop', 75000), ('Keyboard', 3000), ('Monitor', 12000), ('Mouse', 500), ('Laptop', 75000)]


In [21]:
total_revenue = sales_rdd.map(lambda x: x[1] * x[2]).reduce(lambda a, b: a + b)
print(total_revenue)

435000


In [22]:
distinct_products = sales_rdd.map(lambda x: x[0]).distinct()
print(distinct_products.collect())

['Laptop', 'Monitor', 'Mouse', 'Keyboard']


In [23]:
qty_gt1 = sales_rdd.filter(lambda x: x[1] > 1)
print(qty_gt1.collect())

[('Mouse', 3, 500), ('Keyboard', 2, 1500), ('Mouse', 2, 500), ('Monitor', 2, 12000), ('Mouse', 3, 500), ('Keyboard', 2, 1500)]


In [24]:
qty_per_product = (sales_rdd
  .map(lambda x: (x[0], x[1]))
  .reduceByKey(lambda a, b: a + b)
)
print(qty_per_product.collect())

[('Laptop', 5), ('Monitor', 4), ('Mouse', 9), ('Keyboard', 5)]


In [25]:
rev_per_product = (sales_rdd
  .map(lambda x: (x[0], x[1] * x[2]))
  .reduceByKey(lambda a, b: a + b)
)
print(rev_per_product.collect())

[('Laptop', 375000), ('Monitor', 48000), ('Mouse', 4500), ('Keyboard', 7500)]


In [26]:
top_product = (sales_rdd
  .map(lambda x: (x[0], x[1] * x[2]))
  .reduceByKey(lambda a, b: a + b)
  .sortBy(lambda x: x[1], ascending=False)
  .take(1)
)
print(top_product)

[('Laptop', 375000)]


In [27]:
# Part 4
names = emp_rdd.map(lambda x: x[0])
print(names.collect())

['Rahul', 'Sneha', 'Arjun', 'Priya', 'Karan', 'Amit', 'Meera', 'Ravi', 'Neha', 'Vikram']


In [28]:
high_earners = emp_rdd.filter(lambda x: x[2] > 70000)
print(high_earners.collect())

[('Arjun', 'IT', 75000), ('Priya', 'Finance', 80000), ('Meera', 'Finance', 82000), ('Ravi', 'IT', 72000), ('Vikram', 'Finance', 90000)]


In [29]:
depts = emp_rdd.map(lambda x: x[1]).distinct()
print(depts.collect())

['IT', 'HR', 'Finance']


In [31]:
total_sal = (emp_rdd
  .map(lambda x: (x[1], x[2]))
  .reduceByKey(lambda a, b: a + b)
)
print(total_sal.collect())

[('IT', 267000), ('HR', 179000), ('Finance', 252000)]


In [30]:
avg_sal = (emp_rdd
  .map(lambda x: (x[1], (x[2], 1)))
  .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
  .map(lambda x: (x[0], x[1][0] / x[1][1]))
)
print(avg_sal.collect())

[('IT', 66750.0), ('HR', 59666.666666666664), ('Finance', 84000.0)]


In [32]:
top_emp = emp_rdd.sortBy(lambda x: x[2], ascending=False).take(1)
print(top_emp)

[('Vikram', 'Finance', 90000)]


In [33]:
# Part 5
emp_count = (emp_rdd
  .map(lambda x: (x[1], 1))
  .reduceByKey(lambda a, b: a + b)
)
print(emp_count.collect())

[('IT', 4), ('HR', 3), ('Finance', 3)]


In [34]:
sorted_emp = emp_rdd.sortBy(lambda x: x[2], ascending=False)
print(sorted_emp.collect())

[('Vikram', 'Finance', 90000), ('Meera', 'Finance', 82000), ('Priya', 'Finance', 80000), ('Arjun', 'IT', 75000), ('Ravi', 'IT', 72000), ('Rahul', 'IT', 70000), ('Neha', 'HR', 61000), ('Sneha', 'HR', 60000), ('Amit', 'HR', 58000), ('Karan', 'IT', 50000)]


In [35]:
top3 = emp_rdd.sortBy(lambda x: x[2], ascending=False).take(3)
print(top3)

[('Vikram', 'Finance', 90000), ('Meera', 'Finance', 82000), ('Priya', 'Finance', 80000)]


In [36]:
dept_name = emp_rdd.map(lambda x: (x[1], x[0]))
print(dept_name.collect())

[('IT', 'Rahul'), ('HR', 'Sneha'), ('IT', 'Arjun'), ('Finance', 'Priya'), ('IT', 'Karan'), ('HR', 'Amit'), ('Finance', 'Meera'), ('IT', 'Ravi'), ('HR', 'Neha'), ('Finance', 'Vikram')]


In [37]:
grouped = (emp_rdd
  .map(lambda x: (x[1], x[0]))
  .groupByKey()
  .map(lambda x: (x[0], list(x[1])))
)
print(grouped.collect())

[('IT', ['Rahul', 'Arjun', 'Karan', 'Ravi']), ('HR', ['Sneha', 'Amit', 'Neha']), ('Finance', ['Priya', 'Meera', 'Vikram'])]


In [38]:
# Part 6
top_dept = (emp_rdd
  .map(lambda x: (x[1], x[2]))
  .reduceByKey(lambda a, b: a + b)
  .sortBy(lambda x: x[1], ascending=False)
  .take(1)
)
print(top_dept)

[('IT', 267000)]


In [41]:
dept_stats = (emp_rdd
  .map(lambda x: (x[1], (x[2], 1)))
  .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
  .map(lambda x: (x[0], x[1][1], x[1][0]))
)
print(dept_stats.collect())

[('IT', 4, 267000), ('HR', 3, 179000), ('Finance', 3, 252000)]


In [42]:
# Final Mini Project (RDD)
rev = (sales_rdd
  .map(lambda x: (x[0], x[1] * x[2]))
  .reduceByKey(lambda a, b: a + b)
)
print(rev.collect())

[('Laptop', 375000), ('Monitor', 48000), ('Mouse', 4500), ('Keyboard', 7500)]


In [43]:
top_seller = (sales_rdd
  .map(lambda x: (x[0], x[1]))
  .reduceByKey(lambda a, b: a + b)
  .sortBy(lambda x: x[1], ascending=False)
  .take(1)
)
print(top_seller)

[('Mouse', 9)]


In [44]:
top_rev = rev.sortBy(lambda x: x[1], ascending=False).take(1)
print(top_rev)

[('Laptop', 375000)]


In [45]:
sold_gt3 = (sales_rdd
  .map(lambda x: (x[0], 1))
  .reduceByKey(lambda a, b: a + b)
  .filter(lambda x: x[1] > 3)
)
print(sold_gt3.collect())

[('Laptop', 5), ('Mouse', 4)]


In [46]:
sorted_rev = rev.sortBy(lambda x: x[1], ascending=False)
print(sorted_rev.collect())

[('Laptop', 375000), ('Monitor', 48000), ('Keyboard', 7500), ('Mouse', 4500)]
